# Phase 2: Telemetry Data Exploration & Validation
This notebook inspects the synthetic telemetry dataset, visualizes metric shifts across failure modes, and establishes statistical baselines.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Ensure project root is accessible
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import PROCESSED_DATA_DIR, SYNTHETIC_DATA_DIR
print("Processed data dir:", PROCESSED_DATA_DIR)

## 1. Load Processed Telemetry Data

In [ ]:
metrics_path = PROCESSED_DATA_DIR / "clean_metrics.csv"
df_metrics = pd.read_csv(metrics_path)
df_metrics['timestamp'] = pd.to_datetime(df_metrics['timestamp'])
print("Metrics shape:", df_metrics.shape)
df_metrics.head()

In [ ]:
logs_path = PROCESSED_DATA_DIR / "clean_logs.jsonl"
logs = []
with open(logs_path, 'r', encoding='utf-8') as f:
    for line in f:
        logs.append(json.loads(line))
df_logs = pd.DataFrame(logs)
df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])
print("Logs shape:", df_logs.shape)
df_logs.head()

## 2. Log Distribution by Service and Level

In [ ]:
pd.crosstab(df_logs['service_name'], df_logs['level'])

## 3. Metric Behavior Across Scenarios

In [ ]:
# Database connection saturation in database service
db_conns = df_metrics[(df_metrics['service_name'] == 'database') & (df_metrics['metric_name'] == 'database_connections_percent')]
print("Database Connection Mean by Scenario:")
print(db_conns.groupby('scenario')['metric_value'].agg(['mean', 'max', 'std']))

In [ ]:
# API Gateway Latency in payment_failure & network_failure
gw_lat = df_metrics[(df_metrics['service_name'] == 'api_gateway') & (df_metrics['metric_name'] == 'api_latency_ms')]
print("API Gateway Latency Mean by Scenario:")
print(gw_lat.groupby('scenario')['metric_value'].agg(['mean', 'max', 'std']))